# Lecture 11 Activity – Which street has the most fatal crashes?

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

In lecture 10's Activity 1, `GROUP BY street` ranked the streets by their
fatal crashes. It counted spellings, not streets: `NORTHERN BOULEVARD`,
`NORTHERN BLVD`, and `Northern Blvd` were three rows, and so were
`GRAND CENTRAL PKWY` and `GRAND CENTRAL PARKWAY`. A ranking of streets tells a
reader where to look for dangerous roads, so a spelling should not move a
street down the list.

> **Once the spellings of each street are merged, which street has the most
> fatal crashes, and does the top ten change?**

### How to work

Work in pairs for 30 minutes. Submit a cleaned street column, a before-and-after
table of every value your rules changed, a new top ten, and a short
interpretation. If you run out of time, finish Task 6 after class.

- The row unit is **one fatal crash**, with the street name that the police
  recorded (`street`, built in lecture 10 from `on_street_name`).
- The setup reads `data/derived/traffic.duckdb`, which lecture 10's notebook
  builds. If the setup prints `False`, run lecture 10's notebook first.
- Each rule rewrites values. Print what it changed before you count anything.
- In the starter cells, replace each `...` with code.

In [ ]:
import re
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option('display.max_rows', 30)

DERIVED = Path('../../data/derived')
db_file = DERIVED / 'traffic.duckdb'
print(db_file.exists())   # False: run lecture 10's notebook first

con = duckdb.connect(str(db_file), read_only=True)
crashes = con.sql('SELECT collision_id, street FROM crashes').df()
con.close()

# Check: 3,469 crashes, one row each.
crashes.shape

## Task 1 · Rank the streets as recorded

Count the different street values and the crashes with no street. Then list
the ten values with the most crashes, as lecture 10 did. Finally, list every
spelling that contains `NORTHERN B`, capitals ignored, with its count.

How many crashes does Northern Boulevard have once its spellings are added
up, and where would that put it in the stored top ten?



<details><summary>Hint</summary>

`.nunique()`, `.isna().sum()`, and `.value_counts().head(10)`. For the
spellings, `.str.contains('NORTHERN B', case=False, na=False)` inside `.loc`.

</details>

In [ ]:
crashes['street'].nunique(), crashes['street'].isna().sum()

In [ ]:
crashes['street'].value_counts().head(...)

In [ ]:
northern = crashes['street'].str.contains(..., case=False, na=False)
crashes.loc[northern, 'street'].value_counts()

*Your answer:*

## Task 2 · Standardize capitals and spaces

Make a column `street_clean`: the street in capitals, with every run of white
space replaced by one space, and no space at either end. How many different
values remain? Which street in the top ten gained a crash?



<details><summary>Hint</summary>

Chain three methods: `.str.upper()`, `.str.replace(r'\s+', ' ', regex=True)`,
and `.str.strip()`. Compare `.value_counts().head(10)` with Task 1.

</details>

In [ ]:
crashes['street_clean'] = (crashes['street'].str.upper()
                           .str.replace(..., ' ', regex=True)
                           .str.strip())
crashes['street_clean'].nunique()

*Your answer:*

## Task 3 · Compare plain text with `\bST\b` in `.str.replace`

Replace `ST` with `STREET` in `street_clean` two ways: with `.str.replace`
as plain text, then with the pattern `\bST\b`. For each version, count the
different values that changed, and show a few of them before and after.

What goes wrong with plain text? Which value does `\bST\b` still get wrong?



<details><summary>Hint</summary>

`.str.replace('ST', 'STREET', regex=False)` replaces the letters wherever
they appear. A changed value is one where `before != after` and `before` is
not missing. `pd.DataFrame({'before': ..., 'after': ...}).drop_duplicates()`
makes the table. For the last question, look for an `ST` at the start of a
name.

</details>

In [ ]:
before = crashes['street_clean']
plain = before.str.replace('ST', 'STREET', regex=False)
changed = before.notna() & (before != plain)
print(before[changed].nunique(), 'values changed')
pd.DataFrame({'before': before[changed],
              'after': plain[changed]}).drop_duplicates().head(6)

In [ ]:
bounded = before.str.replace(..., 'STREET', regex=True)
changed = ...
print(before[changed].nunique(), 'values changed')
table = pd.DataFrame({'before': before[changed],
                      'after': bounded[changed]}).drop_duplicates()
table

*Your answer:*

## Task 4 · Expand the abbreviations with a dictionary of rules

Complete the dictionary `rules`, which maps a pattern to the full word. The
loop below it applies every rule in order to `street_clean`, as the lecture's
loop labeled each vehicle, and stores the result as `street_full`. Then:

1. Add the rule that turns `ST` at the **start** of a name into `SAINT`, before
   the `STREET` rule.
2. Add rules that turn `E`, `W`, `N`, and `S` into `EAST`, `WEST`, `NORTH`,
   and `SOUTH`, **only at the start** of a name.
3. Count the crashes that changed and the different values that remain.
   Then print, before and after, every changed value that your new rules
   touched: `ST` or a compass letter at the start of the name.
4. List the values of `street_clean` in which `S` is a whole word, with
   their `street_full`. Which of them changed?

Why must the compass rules use `^`? Look at `AVENUE S` and
`THOMAS S BOYLAND STREET` in the last table.



<details><summary>Hint</summary>

`^` matches the start of the string, so `r'^ST\b'` is `ST` as the first word.
A Python dictionary keeps the order in which you write its keys, so put the
`SAINT` rule first. A changed crash is one where `street_clean` is not missing
and differs from `street_full`. For the last table, `\bS\b` is `S` as a
whole word, and `.loc` with a list of two columns keeps both.

</details>

In [ ]:
rules = {
    ...: 'SAINT',
    r'\bST\b': 'STREET',
    r'\bAVE\b': 'AVENUE',
    r'\bBLVD\b': 'BOULEVARD',
    r'\bPKWY\b': 'PARKWAY',
    r'\bEXPY\b|\bEXPWY\b': 'EXPRESSWAY',
    r'\bRD\b': 'ROAD',
    r'\bDR\b': 'DRIVE',
    r'\bPL\b': 'PLACE',
    ...: 'EAST', ...: 'WEST', ...: 'NORTH', ...: 'SOUTH',
}

street_full = crashes['street_clean']
for pattern, full in rules.items():
    street_full = street_full.str.replace(pattern, full, regex=True)
crashes['street_full'] = street_full

In [ ]:
changed = ...
print(changed.sum(), 'crashes changed;',
      crashes['street_full'].nunique(), 'different values remain')
new_rules = crashes['street_clean'].str.contains(..., na=False)
(crashes.loc[changed & new_rules, ['street_clean', 'street_full']]
 .drop_duplicates().sort_values('street_clean'))

In [ ]:
single_s = crashes['street_clean'].str.contains(..., na=False)
(crashes.loc[single_s, ['street_clean', 'street_full']]
 .drop_duplicates())

*Your answer:*

## Task 5 · Rank the cleaned streets and decide two merges

1. List the twelve values of `street_full` with the most crashes. Which street
   entered the top ten, and which left it?
2. Some values name a ramp, an exit, or a service road beside a highway.
   Count the crashes whose `street_full` contains one of the words `RAMP`,
   `SERVICE`, `SVC`, `EXIT`, or `ENTRANCE`, and list them.
3. Third Avenue still has three spellings. List the values that contain
   `3 AVENUE`, `3RD AVENUE`, or `THIRD AVENUE`. Why is
   `.str.contains('3 AVENUE')` the wrong test?

Would you merge the ramps with their highway, and the spellings of Third
Avenue? Say how each choice changes the top of the list.



<details><summary>Hint</summary>

Use one pattern with `\b` around a group of alternatives:
`r'\b(?:RAMP|SERVICE|SVC|EXIT|ENTRANCE)\b'`. For Third Avenue, anchor the
whole value with `^` and `$`, or look at what `3 AVENUE` matches inside
`73 AVENUE`.

</details>

In [ ]:
crashes['street_full'].value_counts().head(...)

In [ ]:
near_highway = crashes['street_full'].str.contains(..., na=False)
print(near_highway.sum(), 'crashes')
crashes.loc[near_highway, 'street_full'].value_counts().head(8)

In [ ]:
third = crashes['street_full'].str.contains(..., na=False)
print(crashes.loc[third, 'street_full'].value_counts())
crashes['street_full'].str.contains('3 AVENUE', na=False).sum()

*Your answer:*

## Task 6 · Interpret the cleaned ranking

Write three or four sentences. Which street has the most fatal crashes after
your rules, and how sure are you of the top three? Name two choices your rules
made, and say how each one moves a street in the ranking.

*Your answer:*